# Level 6 — Tận dụng bộ ba quan hệ RelTR trong sinh chú thích ảnh

Notebook này nối trực tiếp RelTR `(subject, predicate, object)` vào **xếp hạng Top-50 → Top-5** của Level 5A, rồi dùng cùng checkpoint, năm embedding chú thích, Beam Search `k=3`, `max_len=30` và bộ chấm COCO Java. Biến thể thay **lựa chọn caption**; điểm đưa vào gate vẫn là relevance Level 5A. Không train/fine-tune trong notebook này.

**Chạy Validation trước.** Mặc định `SPLIT='val'`, `BETA=0.2` cố định từ prototype. Chỉ đổi sang Test sau khi Val cùng seed hoàn tất và được xem xét; Test có khóa riêng, không xóa khóa. Các ảnh không có quan hệ hoặc không có caption ứng viên khớp quan hệ sẽ giữ nguyên Top-5 đối chứng. Điểm quan hệ là khớp từ vựng có thứ tự S→P→O, chưa phải bộ phân tích cú pháp đầy đủ.

Đây **chưa phải kho tri thức bằng bộ ba**: kho và năm token vẫn là caption embedding cũ. Kết quả Val/Test mới chỉ đo tác dụng của bộ ba ở bước chọn caption.


## 1. Môi trường Colab và Google Drive


In [1]:
import os, sys, subprocess, json, hashlib
from pathlib import Path
from google.colab import drive
sys.dont_write_bytecode = True
drive.mount('/content/drive')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'nltk==3.9.1', 'pycocoevalcap==1.2', 'faiss-cpu'], check=True)
java_home = Path('/usr/lib/jvm/java-11-openjdk-amd64')
if not (java_home/'bin/java').is_file():
    subprocess.run(['apt-get', 'update', '-qq'], check=True)
    subprocess.run(['apt-get', 'install', '-y', '-qq', 'openjdk-11-jre-headless'], check=True)
os.environ['JAVA_HOME'] = str(java_home)
os.environ['PATH'] = str(java_home/'bin') + ':' + os.environ.get('PATH', '')
subprocess.run(['java', '-version'], check=True)
import nltk
if nltk.__version__ != '3.9.1':
    raise RuntimeError('Restart session rồi chạy lại để nạp NLTK 3.9.1')
for corpus in ['wordnet', 'omw-1.4']:
    if not nltk.download(corpus, quiet=True, raise_on_error=True):
        raise RuntimeError('Không tải được ' + corpus)
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Chọn Runtime > Change runtime type > GPU rồi Run all. Sinh caption 5.000 ảnh cần GPU để chạy thực tế.')

Mounted at /content/drive


## 2. Cấu hình thí nghiệm
Sửa `DATA_ROOT` nếu Drive là `/content/drive/MyDrive/Data/Data`. Chọn một seed đã có checkpoint Level 5A.


In [2]:
DATA_ROOT = Path('/content/drive/MyDrive/Data')
SEED = 44
SPLIT = 'test'       # Sau khi xem kết quả Val, mới chủ động đổi thành 'test'.
ALLOW_TEST = True   # Đổi True cùng lúc với SPLIT='test'; không tự chạy Test khi Run all.
BETA = 0.2           # Tỷ trọng điểm quan hệ trong bước chọn Top-5.
MIN_CONFIDENCE = 0.5
MAX_TRIPLETS = 10
OUTPUT_ROOT = DATA_ROOT/'baseline_level6'/'Tan_Dung_RelTR_3Quanhe'
OVERRIDES = {'vocab_path': None, 'annotations_dir': None,
             'feature_dir': None, 'kb_dir': None}
if SPLIT not in ('val', 'test') or (SPLIT == 'test' and not ALLOW_TEST):
    raise ValueError('Mặc định chỉ chạy Val. Để chấm Test, đặt SPLIT=test và ALLOW_TEST=True.')
print('Split:', SPLIT, '| seed:', SEED, '| beta:', BETA)
print('Checkpoint:', DATA_ROOT/'baseline_level5/lv5_A/trained_5tokens_v1'/f'seed{SEED}/best_model.pt')


Split: test | seed: 44 | beta: 0.2
Checkpoint: /content/drive/MyDrive/Data/baseline_level5/lv5_A/trained_5tokens_v1/seed44/best_model.pt


## 3. Xác minh mã Level 5A và nạp module bộ ba
Các module được nhúng trong notebook, xác minh SHA-256 trước khi chạy; không ghi đè mã Level 5A trên Drive.


### Hướng dẫn kiểm tra tệp tin

Vui lòng kiểm tra lại Google Drive của bạn để đảm bảo tệp `multiseed.py` và các tệp khác trong thư mục `baseline_level5/lv5_A/multiseed_5tokens_v1` đã được tải lên đúng vị trí. Bạn có thể sử dụng đoạn mã Python dưới đây để kiểm tra sự tồn tại của tệp cụ thể.

In [3]:
EXPECTED_SOURCE_HASHES = {'lv5_A/multiseed_5tokens_v1/multiseed.py': '17d8244b2922f8596d646d3472f5045e78f3b589df7418f7ba923162006e0497', 'lv5_A/multiseed_5tokens_v1/storage.py': 'cad0c9d25491f98e04c27de5ed0e1199bc99b7581e6e134e2fcff831684508b1', 'lv5_A/multiseed_5tokens_v1/level5a_pipeline.py': '5e73694a07427e8e251d1f183c96aea94a247c3d8e19f44e61933701dd4df086', 'lv5_A/multiseed_5tokens_v1/level5_model.py': '004e171a7349de30ee8cd453af680cb0dace33420704e13cefcd9d3e5ba2ae05', 'lv5_A/multiseed_5tokens_v1/coco_eval_worker.py': '7d8cf752e85fbab35a6ff275c514990fa81be0d777c895312625df448d39fab0', 'shared_modules/semantic_expansion.py': '126119e4484b8d0419b853f1752321e59a44f8f65544b74d838368ef4f8eb9fe', 'shared_modules/mmr_diversity.py': 'a3f6944129aefc33ad3e57350c93ad9ddd873e0546eaf0575d42dfeea44cf805'}
BASE = DATA_ROOT/'baseline_level5'
for rel, wanted in EXPECTED_SOURCE_HASHES.items():
    path = BASE/rel
    if not path.is_file(): raise FileNotFoundError(path)
    if hashlib.sha256(path.read_bytes()).hexdigest() != wanted:
        raise RuntimeError(f'Mã đã thay đổi: {path}. Cần kiểm tra phiên bản trước khi so sánh.')
CODE_DIR = BASE/'lv5_A/multiseed_5tokens_v1'
SHARED_DIR = BASE/'shared_modules'
for name in ['level5a_pipeline','level5_model','storage','semantic_expansion','mmr_diversity']:
    sys.modules.pop(name, None)
for directory in [SHARED_DIR, CODE_DIR]:
    sys.path.insert(0, str(directory))

RUNTIME_DIR = Path('/content/lv6_triplet_runtime')
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
EMBEDDED_SOURCES = {
    'caption_ablation_runner.py': '"""L5A inference ablations on Karpathy Test (or Val), using reviewed native interfaces.\n\nNever calls the original constructor/train/test: all outputs live in baseline_level6.\nFull relevance and Top-5 are computed together once and frozen for all three variants.\nOriginal Top-5 is retained for diagnostics, not mixed into the new comparison.\n"""\nimport csv\nimport hashlib\nimport importlib.metadata\nimport json\nimport math\nimport platform\nimport subprocess\nfrom functools import partial\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\nimport nltk\nimport faiss\nimport level5a_pipeline as P\nimport level5_model as M\nimport semantic_expansion as semantic\nimport mmr_diversity as diversity\nimport storage\n\nFOLDERS = {\'full\': \'_baseline\', \'no_mmr\': \'Tach_Dimersity\', \'no_semantic_expansion\': \'Tach_Sematic\',\n           \'no_reltr_reranking\': \'Tach_ReRanking\'}\nMETRICS = [\'Bleu_1\', \'Bleu_2\', \'Bleu_3\', \'Bleu_4\', \'METEOR\', \'ROUGE_L\', \'CIDEr\', \'SPICE\']\n\n\ndef read_json(path):\n    # Plain read: do not run storage recovery against original L5 folders.\n    return json.loads(Path(path).read_text(encoding=\'utf-8\'))\n\n\ndef read_cache(path, expected):\n    """Validate complete cache or its pending copy, without repairing L5 files."""\n    path = Path(path)\n    failures = []\n    for source in [path, path.with_name(path.name + \'.pending\')]:\n        if not source.is_file():\n            continue\n        try:\n            with np.load(source, allow_pickle=False) as z:\n                meta = json.loads(str(z[\'metadata_json\'].item()))\n                arrays = {k: z[k] for k in z.files if k != \'metadata_json\'}\n            if meta[\'cache_sha256\'] != storage.payload_hash(arrays, meta):\n                raise ValueError(\'Invalid payload hash\')\n        except (ValueError, KeyError, OSError, EOFError) as exc:\n            failures.append(str(exc))\n            continue\n        if any(meta.get(k) != v for k, v in expected.items()):\n            raise ValueError(f\'Cache khác checkpoint/split/config: {source}\')\n        return arrays, meta\n    raise FileNotFoundError(f\'Thiếu cache hoàn chỉnh của L5A: {path}; {failures}\')\n\n\ndef select_variant(labels, captions, embeddings, clip_scores, w, lam, variant):\n    if variant not in (\'no_semantic_expansion\', \'no_reltr_reranking\'):\n        raise ValueError(variant)\n    # Same normalization of label strings as expand_labels, no aliases/synonyms.\n    # Do not add lemmatization that was absent from the actual baseline.\n    if variant == \'no_semantic_expansion\' and labels:\n        terms = {s.lower().strip().replace(\'_\', \' \') for s in labels if s.strip()}\n        jac = np.asarray([semantic.jaccard_semantic(terms, c) for c in captions])\n        raw = w * np.asarray(clip_scores) + (1 - w) * jac\n    else:\n        raw = np.asarray(clip_scores, dtype=np.float64)\n    relevance = P.normalize_scores_per_image(raw)\n    candidates = [dict(position=i, relevance=float(relevance[i]), embedding=e)\n                  for i, e in enumerate(embeddings)]\n    chosen = diversity.mmr_select(candidates, k=5, lambda_param=lam)\n    order = np.asarray([c[\'position\'] for c in chosen], dtype=np.int64)\n    if len(order) != 5 or len(set(order.tolist())) != 5:\n        raise ValueError(\'MMR must select exactly five distinct rows\')\n    order = np.asarray(sorted(order, key=lambda i: (-relevance[i], int(i))), dtype=np.int64)\n    return order, relevance[order].astype(np.float32)\n\n\ndef select_no_mmr(relevance):\n    """Remove only MMR, keep precisely the frozen full-baseline relevance."""\n    relevance = np.asarray(relevance, dtype=np.float64)\n    if relevance.ndim != 1 or len(relevance) < 5 or not np.isfinite(relevance).all():\n        raise ValueError(\'Cần ít nhất 5 relevance hữu hạn\')\n    if not ((relevance >= 0) & (relevance <= 1)).all():\n        raise ValueError(\'Relevance phải ở [0,1]\')\n    order = np.asarray(sorted(range(len(relevance)), key=lambda i: (-relevance[i], i))[:5], dtype=np.int64)\n    return order, relevance[order].astype(np.float32)\n\n\ndef checked_metrics(metrics):\n    if set(metrics) != set(METRICS) or not all(math.isfinite(float(x)) for x in metrics.values()):\n        raise ValueError(\'Cần đủ 8 metric COCO hữu hạn; không điền metric thiếu bằng 0.\')\n    return {k: float(metrics[k]) for k in METRICS}\n\n\ndef relative_inputs(hashes, root):\n    # Permit moving a whole Data directory; validate the content hashes, not old prefix.\n    prefix = str(root).replace(\'\\\\\', \'/\').rstrip(\'/\') + \'/\'\n    hashes = {k.replace(\'\\\\\', \'/\'): v for k, v in hashes.items()}\n    if not all(k.startswith(prefix) for k in hashes):\n        raise ValueError(\'Input nằm ngoài data_root; cần kiểm tra ánh xạ đường dẫn.\')\n    return {k[len(prefix):]: v for k, v in hashes.items()}\n\n\nclass CaptionAblation(P.Experiment):\n    def __init__(self, data_root, seed, output_root, runner_sha, overrides=None, split="test"):\n        if split not in ("val", "test"):\n            raise ValueError("SPLIT phải là val hoặc test")\n        self.split = split\n        self.root = Path(data_root)\n        self.seed = int(seed)\n        if self.seed not in (42, 43, 44, 45, 46):\n            raise ValueError(\'Invalid seed\')\n        self.cfg = dict(data_root=str(self.root), seed=self.seed, phase=\'ablation\',\n                        eval_timeout_seconds=7200, reltr_dir=str(self.root/\'reltr_inputs\'))\n        self.cfg.update(overrides or {})\n        self.smoke = False\n        self.original = self.root/\'baseline_level5/lv5_A/trained_5tokens_v1\'/f\'seed{self.seed}\'\n        self.output_root = Path(output_root)\n        if self.output_root.resolve().is_relative_to((self.root/\'baseline_level5\').resolve()):\n            raise ValueError(\'Output không được nằm trong baseline_level5\')\n        checkpoint_path = self.original/\'best_model.pt\'\n        log = read_json(self.original/\'training_log.json\')\n        val = read_json(self.original/\'validation_summary.json\')\n        final_cfg = read_json(self.original/\'final_config.json\')\n        run_manifest = read_json(self.original/\'run_manifest.json\')\n        if log.get(\'complete\') is not True or log.get(\'completed_epochs\') != 15 or log.get(\'optimizer_steps\', 0) <= 0:\n            raise ValueError(\'Seed chưa có bằng chứng hoàn tất train 15 epochs\')\n        if val.get(\'trained\') is not True or val.get(\'training_seed\') != self.seed or val.get(\'split\') != \'val\':\n            raise ValueError(\'Validation summary không thuộc seed đã train\')\n        if final_cfg.get(\'selected_on\') != \'validation_only\':\n            raise ValueError(\'Config phải được chọn trên Validation\')\n        self.w, self.lam = float(final_cfg[\'w\']), float(final_cfg[\'lambda_param\'])\n        if self.w != .6 or self.lam not in P.LAMBDA_GRID:\n            raise ValueError(\'Config ngoài recipe L5A\')\n        if (val[\'w\'], val[\'lambda_param\']) != (self.w, self.lam):\n            raise ValueError(\'Summary và final_config khác tham số\')\n        contract_id = val[\'contract_sha256\']\n        if any(obj.get(\'contract_sha256\') != contract_id for obj in [log, final_cfg, run_manifest]):\n            raise ValueError(\'Hồ sơ seed khác contract\')\n        self.code_hashes = {n: P.sha256_file(Path(P.__file__).parent/n) for n in val[\'code_sha256\']}\n        if self.code_hashes != val[\'code_sha256\']:\n            raise ValueError(\'Mã trên Drive đã khác mã tạo baseline; cần kiểm tra trước khi chạy\')\n        shared_hashes = {Path(m.__file__).name: P.sha256_file(m.__file__) for m in [semantic, diversity]}\n        if shared_hashes != val[\'shared_modules_sha256\']:\n            raise ValueError(\'Shared modules khác baseline\')\n        self.environment = dict(python=platform.python_version(), torch=torch.__version__,\n            numpy=np.__version__, nltk=nltk.__version__, wordnet=nltk.corpus.wordnet.get_version(),\n            pycocoevalcap=importlib.metadata.version(\'pycocoevalcap\'), faiss=faiss.__version__)\n        self.environment[\'java\'] = subprocess.run([\'java\', \'-version\'], capture_output=True,\n                                                  text=True, check=True).stderr\n        if \'version "11.\' not in self.environment[\'java\']:\n            raise ValueError(\'Evaluator cần Java 11\')\n        if any(self.environment[k] != val[\'environment\'][k] for k in [\'nltk\',\'wordnet\',\'pycocoevalcap\']):\n            raise ValueError(\'NLTK/WordNet/evaluator khác phiên bản baseline\')\n        self.device = torch.device(\'cuda\' if torch.cuda.is_available() else \'cpu\')\n        print(\'Device:\', self.device, \'| seed:\', self.seed, \'| lambda:\', self.lam)\n        self.load_data()  # Original read-only loader, checks split IDs, source KB, feature alignment.\n        old_root = str(Path(run_manifest[\'init_checkpoint\']).parents[3])\n        if relative_inputs(self.input_hashes, self.root) != relative_inputs(run_manifest[\'input_sha256\'], old_root):\n            raise ValueError(\'Dữ liệu/feature/KB/RelTR khác lần train L5A\')\n        saved = P.load_torch(checkpoint_path)\n        if saved.get(\'contract\') != log_without_history(log):\n            # Verify the exact actual training_contract fields, excluding log-only evidence.\n            raise ValueError(\'Checkpoint best_model.pt không khớp training_log\')\n        evidence = saved[\'evidence\']\n        state_hash = M.state_dict_sha256(saved[\'model_state_dict\'])\n        if state_hash != evidence[\'model_state_sha256\'] or state_hash != val[\'model_state_sha256\'] or state_hash != final_cfg[\'checkpoint_sha256\']:\n            raise ValueError(\'Checkpoint khác model đã chấm Validation\')\n        if not evidence.get(\'trained\') or evidence.get(\'training_seed\') != self.seed or evidence.get(\'completed_epochs\') != 15:\n            raise ValueError(\'Checkpoint không có bằng chứng train đúng seed\')\n        if state_hash == evidence[\'initial_state_sha256\']:\n            raise ValueError(\'Checkpoint còn nguyên trọng số khởi tạo\')\n        P.set_seed(self.seed)\n        self.model, self.greedy, beam = M.create_level5(self.stoi, self.device)\n        self.model.load_state_dict(saved[\'model_state_dict\'], strict=True)\n        self.model.eval()\n        del saved\n        self.beam = partial(beam, beam_size=3, max_len=30, alpha=.6)\n        self.model_hash = state_hash\n        self.source_cache_root = self.original.parent/\'cache\'/contract_id[:20]\n        self.identity = dict(protocol=\'l6-three-inference-ablations-v4\', seed=self.seed, split=self.split,\n            evaluation_ids_sha256=P.digest(self.ids[self.split]),\n            model_state_sha256=state_hash, checkpoint_file_sha256=P.sha256_file(checkpoint_path),\n            original_contract_sha256=contract_id, w=self.w, lambda_param=self.lam,\n            beam=3, max_len=30, alpha=.6, runner_sha256=runner_sha,\n            code_sha256=self.code_hashes, shared_sha256=shared_hashes,\n            input_sha256=relative_inputs(self.input_hashes, self.root), environment=self.environment,\n            device=str(self.device), gpu=torch.cuda.get_device_name(0) if self.device.type==\'cuda\' else None)\n        self.contract_id = P.digest(self.identity)\n        self.base_meta = self.identity\n        self.common_dir = self.output_root/\'_baseline\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True, exist_ok=True)\n        self.cache_root = self.common_dir/\'cache\'\n        self.source_contract_id = contract_id\n        self.pre_reference_identity = dict(self.identity)\n        self.reference_dir = self.common_dir/\'retrieval_reference\'\n        self.test_record_path = self.output_root/\'_test_protocol\'/f\'seed{self.seed}\'/\'test_once.json\'\n        P.write_json(self.common_dir/\'experiment_manifest.json\', self.identity)\n\n    def register_test(self, reference_hash=None, completed=None, summary_path=None):\n        """Freeze this seed\'s Test protocol; resume unfinished work, never retune."""\n        if self.split != \'test\':\n            return\n        if self.test_record_path.is_file():\n            record=read_json(self.test_record_path)\n            if record[\'identity\'] != self.pre_reference_identity:\n                raise ValueError(\'Test đã khóa với checkpoint/cấu hình/môi trường khác. Không tự mở lượt Test mới; kiểm tra test_once.json.\')\n        else:\n            record=dict(identity=self.pre_reference_identity, reference_sha256=None,\n                        completed_variants={}, registered_variants=list(FOLDERS))\n        if reference_hash is not None:\n            if record[\'reference_sha256\'] not in (None,reference_hash):\n                raise ValueError(\'Snapshot Test đã đổi. Khôi phục snapshot đã khóa; không tính lại để chọn kết quả.\')\n            record[\'reference_sha256\']=reference_hash\n        if completed is not None:\n            checksum=P.sha256_file(summary_path)\n            previous=record[\'completed_variants\'].get(completed)\n            if previous is not None and previous != checksum:\n                raise ValueError(\'Kết quả Test hoàn tất đã bị thay đổi\')\n            record[\'completed_variants\'][completed]=checksum\n        record[\'status\']=\'complete\' if set(record[\'completed_variants\'])==set(FOLDERS) else \'in_progress\'\n        P.write_json(self.test_record_path,record)\n\n    def original_top5(self):\n        path = self.source_cache_root/f\'w{self.w:.1f}_lambda{self.lam:.1f}\'/f\'{self.split}.npz\'\n        arrays, _ = read_cache(path, dict(contract_sha256=self.source_contract_id,\n            split=self.split, w=self.w, lambda_param=self.lam, stage=\'rerank_mmr\',\n            ids_sha256=P.digest(self.ids[self.split])))\n        P.validate_top5(arrays, len(self.kb))\n        if not np.array_equal(arrays[\'image_ids\'], self.ids[self.split]):\n            raise ValueError(\'Baseline cache image IDs khác split đã chọn\')\n        if np.any(self.kb_ids[arrays[\'row_indices\']] == arrays[\'image_ids\'][:, None]):\n            raise ValueError(\'Baseline cache có caption của cùng ảnh truy vấn\')\n        return arrays\n\n    def original_pool(self):\n        chunks = []\n        for start in range(0, len(self.ids[self.split]), 1000):\n            ids = self.ids[self.split][start:start+1000]\n            chunk, _ = read_cache(self.source_cache_root/\'top50\'/self.split/f\'{start:06d}.npz\',\n                dict(contract_sha256=self.source_contract_id, stage=\'faiss_top50\', split=self.split,\n                     ids_sha256=P.digest(ids)))\n            rows, scores = chunk[\'row_indices\'], chunk[\'clip_scores\']\n            if rows.shape != (len(ids), 50) or scores.shape != rows.shape or not np.isfinite(scores).all():\n                raise ValueError(\'Invalid Top-50 shapes/scores\')\n            if not np.issubdtype(rows.dtype, np.integer) or not ((rows >= 0) & (rows < len(self.kb))).all():\n                raise ValueError(\'Invalid KB rows\')\n            if not np.array_equal(chunk[\'image_ids\'], ids) or any(len(set(r)) != 50 for r in rows):\n                raise ValueError(\'Duplicate/misaligned Top-50\')\n            if np.any(self.kb_ids[rows] == np.asarray(ids)[:, None]):\n                raise ValueError(\'Top-50 chứa cùng source image\')\n            chunks.append(chunk)\n        return {k: np.concatenate([c[k] for c in chunks]) for k in chunks[0]}\n\n    def frozen_reference(self, pool, original_base):\n        """Original rerank_one unchanged; freeze ALL 50 scores, not only selected five.\n\n        This avoids a no-MMR intervention accidentally changing WordNet expansion.\n        Subsequent notebooks reuse this exact snapshot; a lost/regenerated snapshot\n        changes the result contract, so old captions/metrics cannot be reused.\n        """\n        path = self.reference_dir/\'reference.npz\'\n        pool_hash = storage.payload_hash(pool, {})\n        expected = dict(stage=\'paired_full_relevance_v4\', split=self.split, pool_sha256=pool_hash,\n                        source_contract_sha256=self.source_contract_id,\n                        w=self.w, lambda_param=self.lam)\n        if path.is_file() or path.with_name(path.name+\'.pending\').is_file():\n            arrays, meta = read_cache(path, expected)\n        else:\n            if self.split==\'test\' and read_json(self.test_record_path)[\'reference_sha256\'] is not None:\n                raise FileNotFoundError(\'Mất snapshot Test đã khóa. Khôi phục file reference.npz; không sinh snapshot mới.\')\n            rows, scores, fallback, all_relevance = [], [], [], []\n            for pos, iid in enumerate(pool[\'image_ids\']):\n                r = pool[\'row_indices\'][pos]\n                selected, sc, _, fb, relevance = P.rerank_one(\n                    self.labels[self.split][int(iid)], [self.kb_captions[j] for j in r],\n                    np.asarray(self.kb[r], dtype=np.float32), pool[\'clip_scores\'][pos],\n                    self.w, self.lam, return_relevance=True)\n                rows.append(r[selected]); scores.append(sc); fallback.append(fb)\n                all_relevance.append(relevance)\n                if (pos+1) % 500 == 0: print(\'Đóng băng relevance full:\',pos+1,\'/\',len(pool[\'image_ids\']))\n            arrays = dict(image_ids=pool[\'image_ids\'].copy(),row_indices=np.asarray(rows,dtype=np.int64),\n                scores=np.asarray(scores,dtype=np.float32),is_fallback=np.asarray(fallback,dtype=bool),\n                full_relevance=np.asarray(all_relevance,dtype=np.float64))\n            P.save_cache(path,arrays,expected)\n            arrays, meta = read_cache(path,expected)\n        P.validate_top5(arrays,len(self.kb))\n        if not np.array_equal(arrays[\'image_ids\'],pool[\'image_ids\']):\n            raise ValueError(\'Reference image IDs mismatch\')\n        relevance = arrays[\'full_relevance\']\n        if relevance.shape != pool[\'row_indices\'].shape or not np.isfinite(relevance).all() or not ((relevance>=0)&(relevance<=1)).all():\n            raise ValueError(\'Invalid frozen relevance\')\n        for pos,(r5,r50) in enumerate(zip(arrays[\'row_indices\'],pool[\'row_indices\'])):\n            positions={int(r):j for j,r in enumerate(r50)}\n            if not set(r5).issubset(positions): raise ValueError(\'Reference Top-5 outside pool\')\n            actual=np.asarray([relevance[pos,positions[int(r)]] for r in r5],dtype=np.float32)\n            if not np.array_equal(actual,arrays[\'scores\'][pos]): raise ValueError(\'Gate scores differ from frozen relevance\')\n        # Hash only content and expected metadata, not cache creation timestamp.\n        reference_hash=storage.payload_hash(arrays,expected)\n        self.register_test(reference_hash=reference_hash)\n        self.identity=dict(self.pre_reference_identity,retrieval_reference_sha256=reference_hash)\n        self.contract_id=P.digest(self.identity);self.base_meta=self.identity\n        self.common_dir=self.output_root/\'_baseline\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True,exist_ok=True)\n        P.write_json(self.common_dir/\'experiment_manifest.json\',self.identity)\n        P.write_json(self.common_dir/\'retrieval_reference_diagnostics.json\',dict(\n            reference_file=str(path),reference_sha256=reference_hash,\n            baseline_policy=\'Full and no-MMR share frozen relevance computed by original rerank_one\',\n            n_images=len(arrays[\'image_ids\']),\n            changed_top5_vs_original=int(np.any(arrays[\'row_indices\']!=original_base[\'row_indices\'],axis=1).sum()),\n            changed_gate_scores_vs_original=int(np.any(arrays[\'scores\']!=original_base[\'scores\'],axis=1).sum())))\n        return arrays\n\n    def make_variant(self, variant, pool, reference=None):\n        rows, scores = [], []\n        for pos, iid in enumerate(pool[\'image_ids\']):\n            r = pool[\'row_indices\'][pos]\n            if variant == \'no_mmr\':\n                if reference is None: raise ValueError(\'No-MMR requires the full frozen relevance\')\n                selected, sc = select_no_mmr(reference[\'full_relevance\'][pos])\n            else:\n                selected, sc = select_variant(self.labels[self.split][int(iid)],\n                    [self.kb_captions[j] for j in r], np.asarray(self.kb[r], dtype=np.float32),\n                    pool[\'clip_scores\'][pos], self.w, self.lam, variant)\n            rows.append(r[selected]); scores.append(sc)\n            if (pos + 1) % 500 == 0:\n                print(variant, pos + 1, \'/\', len(pool[\'image_ids\']))\n        arrays = dict(image_ids=pool[\'image_ids\'].copy(), row_indices=np.asarray(rows, dtype=np.int64),\n            scores=np.asarray(scores, dtype=np.float32),\n            is_fallback=np.asarray([not self.labels[self.split][int(i)] for i in pool[\'image_ids\']], dtype=bool))\n        P.validate_top5(arrays, len(self.kb))\n        return arrays\n\n    def score(self, variant, cache):\n        self.work = self.output_root/FOLDERS[variant]/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.work.mkdir(parents=True, exist_ok=True)\n        summary_path=self.work/\'summary.json\'\n        if summary_path.is_file():\n            summary=read_json(summary_path)\n            metric=read_json(self.work/\'metrics_coco.json\')\n            if (summary[\'identity\']!=self.identity or summary[\'variant\']!=variant\n                or summary[\'split\']!=self.split or summary[\'n_images\']!=len(self.ids[self.split])):\n                raise ValueError(\'Summary khác cấu hình/split/ảnh đang đánh giá\')\n            checked_metrics(summary[\'metrics_raw\'])\n            if (not metric[\'complete\'] or metric[\'metrics\']!=summary[\'metrics_raw\']\n                or any(summary[\'metrics_x100\'].get(m)!=100*summary[\'metrics_raw\'][m] for m in METRICS)):\n                raise ValueError(\'Metric đã lưu không khớp summary\')\n            self.register_test(completed=variant,summary_path=summary_path)\n            print(\'Dùng kết quả hoàn tất, không sinh/chấm lại:\',variant,self.split)\n            return summary\n        # Persist exact decoder inputs and predictions, with resumable prediction chunks.\n        P.save_cache(self.work/\'top5.npz\', cache, self.meta(variant=variant))\n        preds = self.predictions(self.split, cache, \'beam\', self.model_hash, variant)\n        P.write_json(self.work/\'predictions.json\', preds)\n        metrics = checked_metrics(self.evaluate(preds, self.split, self.work/\'metrics_coco.json\'))\n        summary = dict(identity=self.identity, variant=variant, split=self.split, n_images=len(preds),\n            metrics_raw=metrics, metrics_x100={k: 100*v for k, v in metrics.items()},\n            caption_metrics_computed=True, checkpoint=str(self.original/\'best_model.pt\'),\n            limitation=\'Inference ablation on one trained seed; not retraining or a five-seed aggregate.\',\n            fps_computed=False)\n        P.write_json(self.work/\'summary.json\', summary)\n        self.register_test(completed=variant,summary_path=summary_path)\n        return summary\n\n    def run_ablation(self, variant):\n        if variant not in FOLDERS or variant == \'full\':\n            raise ValueError(variant)\n        self.register_test()\n        base = self.original_top5()\n        pool = self.original_pool()\n        for r5, r50 in zip(base[\'row_indices\'], pool[\'row_indices\']):\n            if not set(r5).issubset(set(r50)):\n                raise ValueError(\'Baseline Top-5 nằm ngoài Top-50\')\n        base = self.frozen_reference(pool,base)\n        variant_cache = self.make_variant(variant, pool,base)\n        self.score(\'full\', base)\n        self.score(variant, variant_cache)\n        return self.comparison()\n\n    def comparison(self):\n        summaries = {}\n        for variant, folder in FOLDERS.items():\n            p = self.output_root/folder/f\'seed{self.seed}\'/self.split/self.contract_id[:20]/\'summary.json\'\n            if p.is_file():\n                summary = read_json(p)\n                if summary[\'identity\'] != self.identity:\n                    raise ValueError(\'Không gộp kết quả khác điều kiện\')\n                checked_metrics(summary[\'metrics_raw\'])\n                if any(summary[\'metrics_x100\'].get(m) != 100*summary[\'metrics_raw\'][m] for m in METRICS):\n                    raise ValueError(\'Summary sai thang metric x100\')\n                summaries[variant] = summary\n        rows = []\n        for variant, s in summaries.items():\n            row = {\'variant\': variant, **s[\'metrics_x100\']}\n            row.update({\'delta_\'+m: s[\'metrics_x100\'][m]-summaries[\'full\'][\'metrics_x100\'][m] for m in METRICS})\n            rows.append(row)\n        out = self.common_dir/\'comparison.csv\'\n        with out.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as f:\n            writer = csv.DictWriter(f, fieldnames=[\'variant\']+METRICS+[\'delta_\'+m for m in METRICS])\n            writer.writeheader(); writer.writerows(rows)\n        P.write_json(self.common_dir/\'comparison.json\', dict(identity=self.identity, rows=rows,\n            delta_definition=\'ablation minus full; units are metric x100 points\',\n            available_variants=list(summaries)))\n        print(\'Bảng metric x100; delta = bỏ thành phần - đầy đủ. Số âm là giảm.\')\n        for row in rows:\n            print(row)\n        print(\'Đã lưu:\', out)\n        return rows\n\n\ndef log_without_history(log):\n    extra = {\'history\', \'completed_epochs\', \'complete\', \'optimizer_steps\',\n             \'initial_state_sha256\', \'best_epoch\', \'best_val_cider\'}\n    return {k: v for k, v in log.items() if k not in extra}\n',
    'reltr_relation_reranking.py': '"""Lv6 prototype: ordered lexical relation matching, not a scene-graph parser."""\nimport csv\nimport hashlib\nimport json\nimport math\nimport re\nfrom collections import defaultdict\nfrom pathlib import Path\n\n_WN = None\n_LEMMATIZER = None\n_ALIASES = {\n    \'man\': \'person\', \'woman\': \'person\', \'boy\': \'person\', \'girl\': \'person\',\n    \'child\': \'person\', \'people\': \'person\', \'men\': \'person\', \'women\': \'person\',\n    \'children\': \'person\', \'bike\': \'bicycle\', \'bikes\': \'bicycle\',\n    \'bicycles\': \'bicycle\', \'dogs\': \'dog\', \'cats\': \'cat\', \'horses\': \'horse\',\n    \'riding\': \'ride\', \'rides\': \'ride\', \'ridden\': \'ride\', \'rode\': \'ride\',\n    \'chasing\': \'chase\', \'chases\': \'chase\', \'chased\': \'chase\',\n    \'holding\': \'hold\', \'holds\': \'hold\', \'held\': \'hold\',\n    \'wearing\': \'wear\', \'wears\': \'wear\', \'wore\': \'wear\',\n    \'sitting\': \'sit\', \'sits\': \'sit\', \'sat\': \'sit\',\n    \'standing\': \'stand\', \'stands\': \'stand\', \'stood\': \'stand\',\n    \'playing\': \'play\', \'plays\': \'play\', \'played\': \'play\',\n    \'eating\': \'eat\', \'eats\': \'eat\', \'ate\': \'eat\',\n    \'carrying\': \'carry\', \'carries\': \'carry\', \'carried\': \'carry\',\n}\n_PHRASE_ALIASES = {\n    \'next to\': (\'beside\', \'alongside\'),\n    \'above\': (\'over\',), \'below\': (\'under\', \'beneath\'),\n}\n\n\ndef configure_wordnet(enabled=True):\n    """Fail explicitly when requested WordNet data are unavailable."""\n    global _WN, _LEMMATIZER\n    if not enabled:\n        _WN, _LEMMATIZER = None, None\n        return\n    from nltk.corpus import wordnet\n    from nltk.stem import WordNetLemmatizer\n    wordnet.ensure_loaded()\n    _WN, _LEMMATIZER = wordnet, WordNetLemmatizer()\n\n\ndef canonical_image_id(value):\n    text = str(value).strip().replace(\'\\\\\', \'/\').rsplit(\'/\', 1)[-1]\n    if re.fullmatch(r\'\\d+\', text):\n        return str(int(text))\n    match = re.fullmatch(r\'COCO_(?:train|val|test)2014_(\\d+)\\.jpg\', text, re.I)\n    if match:\n        return str(int(match.group(1)))\n    raise ValueError(f\'image_id/filename COCO không hợp lệ: {value!r}\')\n\n\ndef words(text):\n    result = []\n    for token in re.findall(r"[a-z]+(?:\'[a-z]+)?", str(text).lower().replace(\'_\', \' \')):\n        token = _ALIASES.get(token, token)\n        if _LEMMATIZER is not None:\n            token = _LEMMATIZER.lemmatize(token, pos=\'v\')\n            token = _LEMMATIZER.lemmatize(token, pos=\'n\')\n        result.append(_ALIASES.get(token, token))\n    return tuple(result)\n\n\ndef load_reltr_triplets(csv_path, min_conf=0.5):\n    """Accept real image/prediction/score CSV and split-column CSV; retain empty images."""\n    if not 0 <= min_conf <= 1:\n        raise ValueError(\'min_conf phải nằm trong [0,1]\')\n    table, seen = {}, defaultdict(set)\n    stats = {\'rows\': 0, \'no_relation\': 0, \'below_threshold\': 0, \'duplicates\': 0}\n    with Path(csv_path).open(encoding=\'utf-8-sig\', newline=\'\') as handle:\n        reader = csv.DictReader(handle)\n        names = {name.strip().lower(): name for name in (reader.fieldnames or [])}\n        def pick(*options):\n            return next((names[name] for name in options if name in names), None)\n        image_col = pick(\'image_id\', \'image\', \'file_name\')\n        pred_col = pick(\'prediction\')\n        sub_col, rel_col, obj_col = pick(\'subject\', \'s\'), pick(\'predicate\', \'relation\', \'rel\', \'p\'), pick(\'object\', \'o\')\n        score_col = pick(\'score\', \'confidence\', \'conf\', \'reltr_score\')\n        if not image_col or not score_col or not (pred_col or (sub_col and rel_col and obj_col)):\n            raise ValueError(f\'CSV thiếu schema RelTR hỗ trợ: {reader.fieldnames}\')\n        for row_number, row in enumerate(reader, 2):\n            try:\n                image_id = canonical_image_id(row[image_col])\n                table.setdefault(image_id, [])\n                stats[\'rows\'] += 1\n                score = float(row[score_col])\n                if not math.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError(\'confidence phải hữu hạn và thuộc [0,1]\')\n                if pred_col:\n                    prediction = row[pred_col].strip()\n                    if prediction.lower() == \'no_relation\':\n                        stats[\'no_relation\'] += 1\n                        continue\n                    parts = re.split(r\'\\s+-\\s+\', prediction, maxsplit=2)\n                    if len(parts) != 3:\n                        raise ValueError(\'prediction phải có dạng subject - relation - object\')\n                    subject, predicate, obj = parts\n                else:\n                    subject, predicate, obj = row[sub_col], row[rel_col], row[obj_col]\n                key = tuple(\' \'.join(x.lower().replace(\'_\', \' \').split()) for x in (subject, predicate, obj))\n                if not all(key):\n                    raise ValueError(\'bộ ba có thành phần rỗng\')\n                if score < min_conf:\n                    stats[\'below_threshold\'] += 1\n                    continue\n                if key in seen[image_id]:\n                    stats[\'duplicates\'] += 1\n                    old = next(t for t in table[image_id] if (t[\'subject\'], t[\'predicate\'], t[\'object\']) == key)\n                    old[\'confidence\'] = max(old[\'confidence\'], score)\n                    continue\n                seen[image_id].add(key)\n                table[image_id].append(dict(zip((\'subject\', \'predicate\', \'object\'), key), confidence=score))\n            except (ValueError, TypeError, KeyError, AttributeError) as exc:\n                raise ValueError(f\'{csv_path}, dòng {row_number}: {exc}\') from exc\n    stats[\'images\'] = len(table)\n    stats[\'triplets\'] = sum(map(len, table.values()))\n    return table, stats\n\n\ndef expand_predicate(predicate, max_synonyms=4):\n    """Sorted same-synset lemmas of first verb sense; no troponym-as-synonym expansion."""\n    base = words(predicate)\n    variants = {base}\n    variants.update(words(p) for p in _PHRASE_ALIASES.get(\' \'.join(base), ()))\n    if _WN is not None and len(base) == 1:\n        synsets = _WN.synsets(base[0], pos=_WN.VERB)\n        if synsets:\n            variants.update(words(name) for name in synsets[0].lemma_names())\n    variants.discard(())\n    return (base,) + tuple(sorted(variants - {base})[:max_synonyms])\n\n\ndef phrase_spans(tokens, phrase):\n    size = len(phrase)\n    return [(i, i + size) for i in range(len(tokens) - size + 1)\n            if size and tokens[i:i + size] == phrase]\n\n\ndef compile_triplets(triplets):\n    return [dict(t, subject_tokens=words(t[\'subject\']), object_tokens=words(t[\'object\']),\n                 predicate_variants=expand_predicate(t[\'predicate\'])) for t in triplets]\n\n\ndef relation_score(text, prepared_triplets, max_gap=12):\n    """Ordered S -> P -> O lexical evidence, restricted to one clause. Not full parsing."""\n    matched, total = [], 0.0\n    clauses = re.split(r\'[.!?;]|\\b(?:and|but|while|whereas)\\b\', text.lower())\n    clauses = [words(clause) for clause in clauses]\n    for triplet in prepared_triplets:\n        conf = triplet[\'confidence\']\n        total += conf\n        hit = False\n        for tokens in clauses:\n            if any(t in {\'not\', \'never\', \'without\', \'no\'} or t.endswith("n\'t") for t in tokens):\n                continue\n            ss = phrase_spans(tokens, triplet[\'subject_tokens\'])\n            oo = phrase_spans(tokens, triplet[\'object_tokens\'])\n            pp = [span for variant in triplet[\'predicate_variants\'] for span in phrase_spans(tokens, variant)]\n            hit = any(s1 <= p0 and p1 <= o0 and p0-s1 <= max_gap and o0-p1 <= max_gap\n                      for s0, s1 in ss for p0, p1 in pp for o0, o1 in oo)\n            if hit:\n                break\n        if hit:\n            matched.append(triplet)\n    return (sum(t[\'confidence\'] for t in matched) / total if total else 0.0), [\n        [t[\'subject\'], t[\'predicate\'], t[\'object\']] for t in matched]\n\n\ndef rerank_candidates(candidates, triplets_by_image, alpha=0.2):\n    """Use original normalized L5A baseline_score; alpha=0 preserves scores and stable ranks."""\n    if not 0 <= alpha <= 1:\n        raise ValueError(\'alpha phải nằm trong [0,1]\')\n    groups, seen = defaultdict(list), set()\n    prepared = {}\n    for raw in candidates:\n        item = dict(raw)\n        image_id = canonical_image_id(item[\'image_id\'])\n        caption_id = str(item[\'caption_id\'])\n        if (image_id, caption_id) in seen:\n            raise ValueError(f\'Caption trùng ID trong cùng ảnh: {image_id}/{caption_id}\')\n        seen.add((image_id, caption_id))\n        if not isinstance(item[\'text\'], str) or not item[\'text\'].strip():\n            raise ValueError(\'Caption rỗng/không phải chuỗi\')\n        base = float(item[\'baseline_score\'])\n        if not math.isfinite(base) or not 0 <= base <= 1:\n            raise ValueError(\'baseline_score phải là điểm L5A đã chuẩn hóa trong [0,1]\')\n        if item.get(\'source_image_id\') is not None and canonical_image_id(item[\'source_image_id\']) == image_id:\n            raise ValueError(\'Ứng viên có cùng source_image_id với ảnh truy vấn\')\n        if image_id not in triplets_by_image:\n            raise ValueError(f\'Thiếu bản ghi RelTR cho ảnh {image_id}; kiểm tra split/image_id\')\n        if image_id not in prepared:\n            prepared[image_id] = compile_triplets(triplets_by_image[image_id]) if alpha else []\n        score, matches = relation_score(item[\'text\'], prepared[image_id]) if alpha else (0.0, [])\n        final = base if alpha == 0 or not prepared[image_id] else (1-alpha)*base + alpha*score\n        item.update(image_id=image_id, baseline_score=base, relation_score=score,\n                    final_score=final, matched_triplets=matches)\n        groups[image_id].append(item)\n    for items in groups.values():\n        items.sort(key=lambda item: -item[\'final_score\'])\n    return dict(groups)\n\n\ndef file_sha256(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as handle:\n        for chunk in iter(lambda: handle.read(1024 * 1024), b\'\'):\n            digest.update(chunk)\n    return digest.hexdigest()',
    'triplet_caption_runner.py': '"""Paired Level-5A caption evaluation with ordered RelTR triplet reranking.\n\nThe decoder and five CLIP-text knowledge embeddings are unchanged. Only the\nselection of five captions from the frozen Level-5A Top-50 pool changes.\n"""\n\nimport csv\nimport json\nimport math\nfrom pathlib import Path\n\nimport numpy as np\n\nimport caption_ablation_runner as A\nimport level5a_pipeline as P\nimport mmr_diversity as D\nimport reltr_relation_reranking as R\n\n\nVARIANT = \'triplet_relation\'\nVARIANTS = (\'full\', VARIANT)\n\n\nclass TripletCaptionExperiment(A.CaptionAblation):\n    def __init__(self, data_root, seed, output_root, base_runner_sha,\n                 triplet_runner_sha, parser_sha, beta=0.2, min_confidence=0.5,\n                 max_triplets=10, overrides=None, split=\'val\'):\n        if not 0 <= beta <= 1 or not math.isfinite(beta):\n            raise ValueError(\'beta phải hữu hạn trong [0, 1]\')\n        if not 0 <= min_confidence <= 1 or not math.isfinite(min_confidence):\n            raise ValueError(\'min_confidence phải hữu hạn trong [0, 1]\')\n        if not isinstance(max_triplets, int) or max_triplets < 1:\n            raise ValueError(\'max_triplets phải là số nguyên dương\')\n        self.beta = float(beta)\n        self.min_confidence = float(min_confidence)\n        self.max_triplets = max_triplets\n        super().__init__(data_root, seed, output_root, base_runner_sha,\n                         overrides=overrides, split=split)\n        self.reltr_csv = Path(self.cfg[\'reltr_dir\'])/f\'coco_{split}_reltr_detection.csv\'\n        self.triplets, self.triplet_stats = R.load_reltr_triplets(\n            self.reltr_csv, min_conf=self.min_confidence)\n        expected_ids = {str(i) for i in self.ids[split]}\n        if set(self.triplets) != expected_ids:\n            missing = len(expected_ids - set(self.triplets))\n            extra = len(set(self.triplets) - expected_ids)\n            raise ValueError(f\'RelTR {split} lệch ID ảnh: thiếu {missing}, thừa {extra}\')\n        self.identity = dict(self.identity,\n            protocol=\'l6-triplet-caption-inference-v1\',\n            triplet_runner_sha256=triplet_runner_sha,\n            triplet_parser_sha256=parser_sha,\n            triplet_csv_sha256=P.sha256_file(self.reltr_csv),\n            beta=self.beta, min_confidence=self.min_confidence,\n            max_triplets=self.max_triplets,\n            relation_policy=\'ordered S-P-O lexical match; top-confidence triplets\',\n            intervention=\'Top-5 selection only; gate scores remain Level-5A full relevance\')\n        self.pre_reference_identity = dict(self.identity)\n        self.contract_id = P.digest(self.identity)\n        self.base_meta = self.identity\n        self.common_dir = self.output_root/\'_baseline\'/f\'seed{self.seed}\'/split/self.contract_id[:20]\n        self.common_dir.mkdir(parents=True, exist_ok=True)\n        self.reference_dir = self.common_dir/\'retrieval_reference\'\n        self.cache_root = self.common_dir/\'cache\'\n        self.test_record_path = self.output_root/\'_test_protocol\'/f\'seed{self.seed}\'/\'test_once.json\'\n        P.write_json(self.common_dir/\'experiment_manifest.json\', self.identity)\n        self.selection_path = self.output_root/\'validation_selection\'/f\'seed{self.seed}.json\'\n        if split == \'test\':\n            self.require_validation()\n\n    def selection_contract(self):\n        return dict(seed=self.seed, beta=self.beta,\n                    min_confidence=self.min_confidence, max_triplets=self.max_triplets,\n                    model_state_sha256=self.model_hash,\n                    triplet_runner_sha256=self.identity[\'triplet_runner_sha256\'],\n                    triplet_parser_sha256=self.identity[\'triplet_parser_sha256\'],\n                    beam=3, max_len=30, alpha=0.6,\n                    policy=self.identity[\'intervention\'])\n\n    def require_validation(self):\n        if not self.selection_path.is_file():\n            raise FileNotFoundError(\'Chạy pha Val cùng seed trước khi mở Test: \'\n                                    + str(self.selection_path))\n        record = A.read_json(self.selection_path)\n        if record.get(\'selection_contract\') != self.selection_contract() or \\\n           record.get(\'split\') != \'val\' or record.get(\'n_images\') != 5000:\n            raise ValueError(\'Cấu hình Test khác cấu hình đã đánh giá trên Val\')\n        if not record.get(\'caption_metrics_computed\'):\n            raise ValueError(\'Chưa có điểm caption Val hoàn chỉnh\')\n\n    def register_test(self, reference_hash=None, completed=None, summary_path=None):\n        if self.split != \'test\':\n            return\n        if self.test_record_path.is_file():\n            record = A.read_json(self.test_record_path)\n            if record.get(\'identity\') != self.pre_reference_identity:\n                raise ValueError(\'Test đã khóa với checkpoint/cấu hình khác; giữ nguyên bản khóa\')\n        else:\n            record = dict(identity=self.pre_reference_identity,\n                          reference_sha256=None, completed_variants={},\n                          registered_variants=list(VARIANTS))\n        if reference_hash is not None:\n            if record[\'reference_sha256\'] not in (None, reference_hash):\n                raise ValueError(\'Snapshot relevance Test đã đổi; cần khôi phục bản gốc\')\n            record[\'reference_sha256\'] = reference_hash\n        if completed is not None:\n            if completed not in VARIANTS:\n                raise ValueError(completed)\n            checksum = P.sha256_file(summary_path)\n            old = record[\'completed_variants\'].get(completed)\n            if old is not None and old != checksum:\n                raise ValueError(\'Summary Test hoàn tất đã bị thay đổi\')\n            record[\'completed_variants\'][completed] = checksum\n        record[\'status\'] = (\'complete\' if set(record[\'completed_variants\']) == set(VARIANTS)\n                            else \'in_progress\')\n        P.write_json(self.test_record_path, record)\n\n    def make_triplet_cache(self, pool, reference):\n        rows, gate_scores, fallback = [], [], []\n        changed, signal, nonempty = 0, 0, 0\n        matched_candidates, total_candidates = 0, 0\n        examples = []\n        for pos, iid_raw in enumerate(pool[\'image_ids\']):\n            iid = int(iid_raw)\n            selected_pool_rows = pool[\'row_indices\'][pos]\n            base_relevance = np.asarray(reference[\'full_relevance\'][pos], dtype=np.float64)\n            triplets = sorted(self.triplets[str(iid)],\n                              key=lambda t: (-t[\'confidence\'], t[\'subject\'],\n                                             t[\'predicate\'], t[\'object\']))[:self.max_triplets]\n            prepared = R.compile_triplets(triplets)\n            nonempty += bool(prepared)\n            relation = np.asarray([\n                R.relation_score(self.kb_captions[int(row)], prepared)[0]\n                for row in selected_pool_rows], dtype=np.float64)\n            if not np.isfinite(relation).all() or np.any((relation < 0) | (relation > 1)):\n                raise ValueError(\'Điểm quan hệ không hợp lệ\')\n            matched_candidates += int(np.count_nonzero(relation))\n            total_candidates += len(relation)\n            has_signal = bool(prepared) and bool(np.any(relation > 0))\n            signal += has_signal\n            # With zero relation evidence, preserve the full baseline exactly.\n            blended = ((1 - self.beta)*base_relevance + self.beta*relation\n                       if has_signal and self.beta else base_relevance)\n            embeddings = np.asarray(self.kb[selected_pool_rows], dtype=np.float32)\n            candidates = [dict(position=i, relevance=float(blended[i]), embedding=embeddings[i])\n                          for i in range(len(blended))]\n            chosen = D.mmr_select(candidates, k=5, lambda_param=self.lam)\n            order = sorted((int(c[\'position\']) for c in chosen),\n                           key=lambda i: (-base_relevance[i], i))\n            if len(order) != 5 or len(set(order)) != 5:\n                raise ValueError(\'Triplet reranking phải trả đúng 5 caption khác nhau\')\n            row = selected_pool_rows[order]\n            differs = set(map(int, row)) != set(map(int, reference[\'row_indices\'][pos]))\n            changed += differs\n            if differs and len(examples) < 10:\n                examples.append(dict(image_id=iid, triplets=triplets,\n                    full_top5=[self.kb_captions[int(r)] for r in reference[\'row_indices\'][pos]],\n                    triplet_top5=[dict(caption=self.kb_captions[int(selected_pool_rows[j])],\n                                       relation_score=float(relation[j])) for j in order]))\n            rows.append(row)\n            # Decoder gate sees the same original relevance scale for each chosen caption.\n            gate_scores.append(base_relevance[order].astype(np.float32))\n            fallback.append(not has_signal)\n            if (pos + 1) % 500 == 0:\n                print(\'Triplet reranking:\', pos + 1, \'/\', len(pool[\'image_ids\']))\n        cache = dict(image_ids=pool[\'image_ids\'].copy(),\n                     row_indices=np.asarray(rows, dtype=np.int64),\n                     scores=np.asarray(gate_scores, dtype=np.float32),\n                     is_fallback=np.asarray(fallback, dtype=bool))\n        P.validate_top5(cache, len(self.kb))\n        diagnostics = dict(split=self.split, seed=self.seed, n_images=len(rows),\n                           n_images_with_triplets=nonempty,\n                           n_images_with_candidate_relation_match=signal,\n                           changed_top5_image_count=changed,\n                           matched_candidate_count=matched_candidates,\n                           total_candidate_count=total_candidates,\n                           reltr_csv_stats=self.triplet_stats,\n                           changed_examples=examples)\n        return cache, diagnostics\n\n    def score(self, variant, cache):\n        if variant == \'full\':\n            return super().score(variant, cache)\n        if variant != VARIANT:\n            raise ValueError(variant)\n        self.work = self.output_root/\'triplet_relation\'/f\'seed{self.seed}\'/self.split/self.contract_id[:20]\n        self.work.mkdir(parents=True, exist_ok=True)\n        summary_path = self.work/\'summary.json\'\n        if summary_path.is_file():\n            summary = A.read_json(summary_path)\n            metric = A.read_json(self.work/\'metrics_coco.json\')\n            if (summary.get(\'identity\') != self.identity or summary.get(\'variant\') != VARIANT\n                or summary.get(\'split\') != self.split\n                or summary.get(\'n_images\') != len(self.ids[self.split])):\n                raise ValueError(\'Summary triplet khác contract/split/ảnh\')\n            A.checked_metrics(summary[\'metrics_raw\'])\n            if not metric.get(\'complete\') or metric.get(\'metrics\') != summary[\'metrics_raw\']:\n                raise ValueError(\'Metrics triplet đã lưu không khớp summary\')\n            self.register_test(completed=VARIANT, summary_path=summary_path)\n            print(\'Dùng kết quả triplet đã hoàn tất; không chấm lại:\', self.split)\n            return summary\n        P.save_cache(self.work/\'top5.npz\', cache, self.meta(variant=VARIANT))\n        predictions = self.predictions(self.split, cache, \'beam\', self.model_hash, VARIANT)\n        P.write_json(self.work/\'predictions.json\', predictions)\n        metrics = A.checked_metrics(self.evaluate(predictions, self.split,\n                                                 self.work/\'metrics_coco.json\'))\n        summary = dict(identity=self.identity, variant=VARIANT, split=self.split,\n                       n_images=len(predictions), metrics_raw=metrics,\n                       metrics_x100={m: 100*value for m, value in metrics.items()},\n                       caption_metrics_computed=True,\n                       checkpoint=str(self.original/\'best_model.pt\'),\n                       trained_during_experiment=False,\n                       limitation=\'Inference-only triplet reranking; not a triplet knowledge base.\',\n                       fps_computed=False)\n        P.write_json(summary_path, summary)\n        self.register_test(completed=VARIANT, summary_path=summary_path)\n        return summary\n\n    def run_triplet(self):\n        self.register_test()\n        original = self.original_top5()\n        pool = self.original_pool()\n        reference = self.frozen_reference(pool, original)\n        cache, diagnostics = self.make_triplet_cache(pool, reference)\n        P.write_json(self.common_dir/\'triplet_diagnostics.json\', diagnostics)\n        if self.beta > 0 and not diagnostics[\'n_images_with_candidate_relation_match\']:\n            raise RuntimeError(\'Không ứng viên nào khớp bộ ba; kiểm tra CSV/nhãn trước khi chấm COCO\')\n        full = self.score(\'full\', reference)\n        triplet = self.score(VARIANT, cache)\n        rows = []\n        for item in (full, triplet):\n            row = dict(variant=item[\'variant\'], **item[\'metrics_x100\'])\n            row.update({\'delta_\'+m: item[\'metrics_x100\'][m]-full[\'metrics_x100\'][m]\n                        for m in A.METRICS})\n            rows.append(row)\n        path = self.common_dir/\'comparison.csv\'\n        with path.open(\'w\', encoding=\'utf-8-sig\', newline=\'\') as handle:\n            writer = csv.DictWriter(handle,\n                fieldnames=[\'variant\']+A.METRICS+[\'delta_\'+m for m in A.METRICS])\n            writer.writeheader()\n            writer.writerows(rows)\n        P.write_json(self.common_dir/\'comparison.json\',\n                     dict(identity=self.identity, rows=rows, diagnostics=diagnostics,\n                          delta_definition=\'triplet_relation minus full; metric x100 points\'))\n        if self.split == \'val\':\n            selection = dict(selection_contract=self.selection_contract(), split=\'val\',\n                             n_images=len(self.ids[\'val\']), caption_metrics_computed=True,\n                             val_cider_full_x100=full[\'metrics_x100\'][\'CIDEr\'],\n                             val_cider_triplet_x100=triplet[\'metrics_x100\'][\'CIDEr\'],\n                             comparison_path=str(path))\n            if self.selection_path.is_file():\n                old = A.read_json(self.selection_path)\n                if old[\'selection_contract\'] != selection[\'selection_contract\']:\n                    raise ValueError(\'Val đã có beta/config khác; không ghi đè lựa chọn cũ\')\n            P.write_json(self.selection_path, selection)\n        return rows, diagnostics\n',
}
SOURCE_SHA256 = {name: hashlib.sha256(src.encode('utf-8')).hexdigest() for name, src in EMBEDDED_SOURCES.items()}
for name, source in EMBEDDED_SOURCES.items():
    path = RUNTIME_DIR/name
    path.write_text(source, encoding='utf-8')
    if hashlib.sha256(path.read_bytes()).hexdigest() != SOURCE_SHA256[name]:
        raise RuntimeError('Mã runtime bị thay đổi: '+name)
sys.path.insert(0, str(RUNTIME_DIR))
for name in ['caption_ablation_runner', 'reltr_relation_reranking', 'triplet_caption_runner']:
    sys.modules.pop(name, None)
import reltr_relation_reranking as R
from triplet_caption_runner import TripletCaptionExperiment
R.configure_wordnet(True)
print('Mã đã xác minh:', SOURCE_SHA256)


Mã đã xác minh: {'caption_ablation_runner.py': '08df488c446a0bb3a75d192d1c7dac40b2ed4f3af728a2192da5f5e1cd03fca4', 'reltr_relation_reranking.py': 'bf85f034b53df7bb5a3418e278aa8fc578ba7b816165e35ec8b53c9b6cee3348', 'triplet_caption_runner.py': '83859183f95bd90c91631af21e6e4ed1ed23238cfc6e4d58b014769537529b8e'}


In [4]:
import math
import warnings
import reltr_relation_reranking as R

_original_relation_score = R.relation_score

def relation_score_safe(text, prepared_triplets, max_gap=12):
    score, matches = _original_relation_score(
        text, prepared_triplets, max_gap=max_gap
    )
    score = float(score)

    # Dữ liệu thực sự hỏng vẫn phải dừng
    if not math.isfinite(score):
        raise ValueError(f"Điểm quan hệ NaN/Inf: {score!r}")

    # Chỉ sửa sai số rất nhỏ quanh [0, 1]
    if score < -1e-12 or score > 1.0 + 1e-12:
        raise ValueError(f"Điểm quan hệ không hợp lệ: {score!r}")

    if score < 0.0 or score > 1.0:
        warnings.warn(f"Hiệu chỉnh sai số số thực: {score!r}")
        score = min(1.0, max(0.0, score))

    return score, matches

R.relation_score = relation_score_safe
print("Đã áp dụng bản vá relation_score.")

Đã áp dụng bản vá relation_score.


## 4. Nạp checkpoint và dữ liệu đúng split
Bước này kiểm tra hồ sơ train 15 epochs, checkpoint, feature, caption KB, RelTR CSV và ID của 5.000 ảnh.


In [5]:
experiment = TripletCaptionExperiment(
    DATA_ROOT, SEED, OUTPUT_ROOT,
    SOURCE_SHA256['caption_ablation_runner.py'],
    SOURCE_SHA256['triplet_caption_runner.py'],
    SOURCE_SHA256['reltr_relation_reranking.py'],
    beta=BETA, min_confidence=MIN_CONFIDENCE,
    max_triplets=MAX_TRIPLETS, overrides=OVERRIDES, split=SPLIT)
print('Model SHA-256:', experiment.model_hash)
print('RelTR CSV:', experiment.reltr_csv)
print('Triplets sau lọc:', experiment.triplet_stats)


Device: cuda | seed: 44 | lambda: 0.7
SHA-256: /content/drive/MyDrive/Data/baseline_level1/vocab.json
SHA-256: /content/drive/MyDrive/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_train.json
SHA-256: /content/drive/MyDrive/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_val.json
SHA-256: /content/drive/MyDrive/Data/COCO_2014/annotations_trainval2014/annotations/captions_karpathy_test.json
SHA-256: /content/drive/MyDrive/Data/knowledge_base_level1_karpathy/text_embeddings_train_restval.npy
SHA-256: /content/drive/MyDrive/Data/knowledge_base_level1_karpathy/text_embedding_mapping.jsonl


Validate KB / FAISS:   0%|          | 0/70 [00:00<?, ?it/s]

SHA-256: /content/drive/MyDrive/Data/baseline_level1/clip_image_features/train_image_ids.json
SHA-256: /content/drive/MyDrive/Data/baseline_level1/clip_image_features/train_features.npy
SHA-256: /content/drive/MyDrive/Data/reltr_inputs/coco_train_reltr_detection.csv
SHA-256: /content/drive/MyDrive/Data/baseline_level1/clip_image_features/val_image_ids.json
SHA-256: /content/drive/MyDrive/Data/baseline_level1/clip_image_features/val_features.npy
SHA-256: /content/drive/MyDrive/Data/reltr_inputs/coco_val_reltr_detection.csv
SHA-256: /content/drive/MyDrive/Data/baseline_level1/clip_image_features/test_image_ids.json
SHA-256: /content/drive/MyDrive/Data/baseline_level1/clip_image_features/test_features.npy
SHA-256: /content/drive/MyDrive/Data/reltr_inputs/coco_test_reltr_detection.csv
Dữ liệu đã kiểm tra; chưa có số liệu thực nghiệm Level 5.
Model SHA-256: 7b6d54d7251f872306e18333db4ad8cad0bcdae9d50b0a37e31afaff8d3ed2f1
RelTR CSV: /content/drive/MyDrive/Data/reltr_inputs/coco_test_reltr_de

## 5. Chọn Top-5 theo quan hệ, sinh caption và chấm 8 chỉ số COCO
Đối chứng `full` và biến thể dùng cùng checkpoint/split/Top-50. Có resume theo chunk và lưu riêng khỏi ba ablation đã chạy.


In [6]:
import fcntl
lock_path = Path('/tmp')/('lv6-triplet-seed'+str(SEED)+'-'+SPLIT+'.lock')
with lock_path.open('w') as lock:
    fcntl.flock(lock, fcntl.LOCK_EX | fcntl.LOCK_NB)
    try:
        comparison_rows, triplet_diagnostics = experiment.run_triplet()
    finally:
        fcntl.flock(lock, fcntl.LOCK_UN)


Đóng băng relevance full: 500 / 5000
Đóng băng relevance full: 1000 / 5000
Đóng băng relevance full: 1500 / 5000
Đóng băng relevance full: 2000 / 5000
Đóng băng relevance full: 2500 / 5000
Đóng băng relevance full: 3000 / 5000
Đóng băng relevance full: 3500 / 5000
Đóng băng relevance full: 4000 / 5000
Đóng băng relevance full: 4500 / 5000
Đóng băng relevance full: 5000 / 5000
Triplet reranking: 500 / 5000
Triplet reranking: 1000 / 5000


/tmp/ipykernel_660/4115131215.py:22: UserWarning: Hiệu chỉnh sai số số thực: 1.0000000000000002
  warnings.warn(f"Hiệu chỉnh sai số số thực: {score!r}")


Triplet reranking: 1500 / 5000
Triplet reranking: 2000 / 5000
Triplet reranking: 2500 / 5000
Triplet reranking: 3000 / 5000
Triplet reranking: 3500 / 5000
Triplet reranking: 4000 / 5000
Triplet reranking: 4500 / 5000
Triplet reranking: 5000 / 5000


test beam:   0%|          | 0/79 [00:00<?, ?it/s]

Đang chấm COCO Java; log: /content/drive/MyDrive/Data/baseline_level6/Tan_Dung_RelTR_3Quanhe/_baseline/seed44/test/bea28859e71747668df4/metrics_coco.log


test beam:   0%|          | 0/79 [00:00<?, ?it/s]

Đang chấm COCO Java; log: /content/drive/MyDrive/Data/baseline_level6/Tan_Dung_RelTR_3Quanhe/triplet_relation/seed44/test/bea28859e71747668df4/metrics_coco.log


## 6. Xem kết quả và vị trí lưu
Metric ×100; delta = `triplet_relation − full`. Đây là kết quả của **một seed**.


In [7]:
import pandas as pd
from IPython.display import display
display(pd.DataFrame(comparison_rows).round(4))
print('Ảnh có triplet:', triplet_diagnostics['n_images_with_triplets'])
print('Ảnh có caption khớp quan hệ:', triplet_diagnostics['n_images_with_candidate_relation_match'])
print('Ảnh đổi bộ Top-5:', triplet_diagnostics['changed_top5_image_count'])
print('Comparison:', experiment.common_dir/'comparison.json')
print('Predictions/metrics:', experiment.output_root/'triplet_relation'/f'seed{SEED}'/SPLIT/experiment.contract_id[:20])
if SPLIT == 'val':
    print('Cấu hình Val cho Test:', experiment.selection_path)


,variant,Bleu_1,Bleu_2,Bleu_3,Bleu_4,METEOR,ROUGE_L,CIDEr,SPICE,delta_Bleu_1,delta_Bleu_2,delta_Bleu_3,delta_Bleu_4,delta_METEOR,delta_ROUGE_L,delta_CIDEr,delta_SPICE
0,full,74.7993,58.4859,44.9437,34.5064,27.1043,55.5533,110.4695,19.7154,0.0000,0.0000,0.000,0.0000,0.0000,0.0000,0.0000,0.0000
1,triplet_relation,74.7775,58.4593,44.9377,34.5283,27.1034,55.5356,110.4024,19.7112,-0.0218,-0.0266,-0.006,0.0219,-0.0009,-0.0177,-0.0671,-0.0042


Ảnh có triplet: 4726
Ảnh có caption khớp quan hệ: 2723
Ảnh đổi bộ Top-5: 823
Comparison: /content/drive/MyDrive/Data/baseline_level6/Tan_Dung_RelTR_3Quanhe/_baseline/seed44/test/bea28859e71747668df4/comparison.json
Predictions/metrics: /content/drive/MyDrive/Data/baseline_level6/Tan_Dung_RelTR_3Quanhe/triplet_relation/seed44/test/bea28859e71747668df4
